In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Loading v2 dataset...")
DATA_PATH = "/Users/darshanv/credit-risk-scorer/data/"
df = pd.read_csv(f"{DATA_PATH}application_train_processed_v2.csv")

# Separate features (X) and target (y)
X = df.drop(columns=['TARGET'])
y = df['TARGET']

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Loading v2 dataset...
Training features shape: (246008, 110)
Testing features shape: (61503, 110)


In [2]:
print("Running 5-Fold Stratified CV on V2 features...")

# Initialize the champion model configuration
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    # Train the model on the current fold
    lgb_model.fit(X_fold_train, y_fold_train)
    
    # Predict probabilities (the [:, 1] gets the probability of class 1: Default)
    val_preds = lgb_model.predict_proba(X_fold_val)[:, 1]
    
    # Evaluate using ROC-AUC
    score = roc_auc_score(y_fold_val, val_preds)
    cv_scores.append(score)
    print(f"  Fold {fold+1}: ROC-AUC = {score:.4f}")

print(f"\nOld Mean ROC-AUC (Phase 5): 0.7548")
print(f"New Mean ROC-AUC (V2): {np.mean(cv_scores):.4f}")
print(f"New Std deviation: {np.std(cv_scores):.4f}")

Running 5-Fold Stratified CV on V2 features...
[LightGBM] [Info] Number of positive: 15888, number of negative: 180918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023623 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10706
[LightGBM] [Info] Number of data points in the train set: 196806, number of used features: 105
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
  Fold 1: ROC-AUC = 0.7690
[LightGBM] [Info] Number of positive: 15888, number of negative: 180918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027366 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10700
[LightGBM] [Info] Number of data 

In [3]:
import joblib

print("Retraining Champion Model on full training set...")
# Fit on the entire 80% X_train
lgb_model.fit(X_train, y_train)

print("Evaluating on test set...")
# Predict probabilities on the 20% X_test
test_preds = lgb_model.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, test_preds)
print(f"Final Test ROC-AUC: {test_roc_auc:.4f}\n")

print("Generating Threshold Analysis...")
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
results = []

for t in thresholds:
    # Convert probabilities to binary predictions based on threshold
    preds_bin = (test_preds >= t).astype(int)
    
    # Calculate True Positives, False Negatives, False Positives, True Negatives
    tp = np.sum((preds_bin == 1) & (y_test == 1))
    fn = np.sum((preds_bin == 0) & (y_test == 1))
    fp = np.sum((preds_bin == 1) & (y_test == 0))
    tn = np.sum((preds_bin == 0) & (y_test == 0))
    
    # Calculate metrics safely
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    ratio = fp / tp if tp > 0 else np.nan
    
    results.append({
        'Threshold': t,
        'Caught (TP)': tp,
        'Missed (FN)': fn,
        'Wrongly Rejected (FP)': fp,
        'Correctly Approved (TN)': tn,
        'Ratio (FP/TP)': round(ratio, 1),
        'Recall': round(recall, 3),
        'Precision': round(precision, 3),
        'F1': round(f1, 4)
    })

# Display the results table
thresh_df = pd.DataFrame(results)
display(thresh_df)

# Save the Champion Model for Phase 7
model_path = "/Users/darshanv/credit-risk-scorer/models/champion_lightgbm_v2.pkl"
joblib.dump(lgb_model, model_path)
print(f"\nSaved v2 champion model to: {model_path}")

Retraining Champion Model on full training set...
[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.029360 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10700
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 106
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Evaluating on test set...
Final Test ROC-AUC: 0.7755

Generating Threshold Analysis...


,Threshold,Caught (TP),Missed (FN),Wrongly Rejected (FP),Correctly Approved (TN),Ratio (FP/TP),Recall,Precision,F1
0,0.10,4945,20,54200,2338,11.0,0.996,0.084,0.1543
1,0.15,4885,80,49304,7234,10.1,0.984,0.090,0.1652
2,0.20,4759,206,43309,13229,9.1,0.959,0.099,0.1795
3,0.25,4592,373,37577,18961,8.2,0.925,0.109,0.1948
4,0.30,4425,540,32177,24361,7.3,0.891,0.121,0.2129
5,0.35,4222,743,27329,29209,6.5,0.850,0.134,0.2312
6,0.40,3966,999,22935,33603,5.8,0.799,0.147,0.2489
7,0.45,3702,1263,18957,37581,5.1,0.746,0.163,0.2680
8,0.50,3418,1547,15467,41071,4.5,0.688,0.181,0.2866



Saved v2 champion model to: /Users/darshanv/credit-risk-scorer/models/champion_lightgbm_v2.pkl


In [4]:
df_v2 = pd.read_csv('/Users/darshanv/credit-risk-scorer/data/application_train_processed_v2.csv')
print(df_v2.shape)
print("\nNew behavioral columns:")
behavioral = [c for c in df_v2.columns if any(c.startswith(p) for p in ['bureau_', 'prev_', 'instal_', 'cc_'])]
print(f"Count: {len(behavioral)}")
print(behavioral)

(307511, 111)

New behavioral columns:
Count: 33
['bureau_loan_count', 'bureau_avg_days_credit', 'bureau_avg_credit_day_overdue', 'bureau_max_days_overdue', 'bureau_total_debt', 'bureau_active_loans', 'bureau_closed_loans', 'bureau_debt_to_credit_ratio', 'bureau_bal_total_records', 'bureau_bal_bad_count', 'bureau_bal_avg_status', 'bureau_bal_bad_ratio', 'prev_app_count', 'prev_approved_count', 'prev_refused_count', 'prev_cancelled_count', 'prev_avg_credit', 'prev_avg_down_payment', 'prev_avg_days_decision', 'prev_approval_rate', 'instal_count', 'instal_avg_days_late', 'instal_max_days_late', 'instal_late_payment_count', 'instal_avg_payment_ratio', 'instal_late_payment_ratio', 'cc_total_records', 'cc_avg_balance', 'cc_max_balance', 'cc_avg_utilization', 'cc_max_utilization', 'cc_avg_payment_ratio', 'cc_late_payment_count']


In [5]:
import joblib

feature_names_v2 = X_train.columns.tolist()
joblib.dump(feature_names_v2,
    '/Users/darshanv/credit-risk-scorer/models/feature_names_v2.pkl')
print(f"Saved {len(feature_names_v2)} feature names")

Saved 110 feature names
